# Tunisian trilingual code-switching evaluation

This notebook runs the reproducible primary evaluation. Replace the lived-experience placeholder in the repository README with your own account before publication.

In [ ]:
import pathlib
import subprocess
import sys
import importlib.util

# Resolve the repository even when Jupyter starts in notebooks/.
REPO = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / 'scripts' / 'run_eval.py').exists()), None)
if REPO is None:
    raise RuntimeError('Could not locate the repository root.')

# Install the notebook/runtime dependencies only when they are missing.
missing = [name for name in ('pandas', 'matplotlib')
           if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                    str(REPO / 'requirements.txt')], cwd=REPO, check=True)

def run_script(script, *args):
    """Run a repository script with the active notebook Python."""
    command = [sys.executable, str(REPO / 'scripts' / script), *map(str, args)]
    return subprocess.run(command, cwd=REPO, check=True)

RAW_OUTPUT = REPO / 'results' / 'raw_outputs.csv'
print(f'Repository: {REPO}')
print(f'Python: {sys.executable}')

## 1. Validate the dataset

The structural validator is necessary but not sufficient. Before publication, add human review for Tunisian naturalness, translation equivalence, and answer correctness.

In [ ]:
run_script('validate_dataset.py')

## 2. Run or load the primary 3B model

This uses deterministic generation. Set `RUN_EVALUATION = True` for a fresh run; it requires a GPU with enough memory and downloads `Qwen/Qwen2.5-3B-Instruct` from Hugging Face. The default uses the repository's cached raw results so the notebook can be run without model downloads.

In [ ]:
# Set this to True for a fresh Qwen2.5-3B evaluation. It requires a working
# CUDA environment and may download several GB from Hugging Face.
RUN_EVALUATION = False

if RUN_EVALUATION:
    run_script(
        'run_eval.py', '--models', 'qwen_3b', '--temperature', '0',
        '--max-new-tokens', '64', '--output', RAW_OUTPUT
    )
else:
    if not RAW_OUTPUT.exists():
        raise FileNotFoundError(f'Raw results not found: {RAW_OUTPUT}')
    print(f'Using existing raw results: {RAW_OUTPUT}')

# A cached artifact may be a partial smoke run; only enforce completeness
# after a fresh evaluation.
if RUN_EVALUATION:
    run_script('check_results.py', '--input', RAW_OUTPUT, '--models', 'qwen_3b')

## 3. Score and plot

Automatic scoring is only a first pass. Inspect the rows marked `needs_manual_review` and a blind sample of automatic matches.

In [ ]:
run_script('score_results.py', '--input', RAW_OUTPUT)
run_script('analyze.py')

In [ ]:
import pandas as pd
from IPython.display import display, Image
scored = pd.read_csv(REPO / 'results' / 'scored_results.csv')
display(scored.groupby(['model', 'condition', 'score']).size().rename('rows').reset_index())
display(scored[['problem_id', 'condition', 'model_response', 'expected_answer', 'score', 'match_type']].sample(min(12, len(scored)), random_state=7))
display(Image(filename=str(REPO / 'results' / 'accuracy_by_condition.png')))

## 4. Qualitative inspection

For the final submission, annotate at least ten responses with an error category: dialect comprehension, code-switch boundary loss, technical reasoning error, unit/format error, instruction-following error, or generation failure. Include representative examples in the written analysis.